In [2]:
# ==========================================
# SQLITE DATABASE CREATION
# ==========================================

In [3]:
pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [4]:
from sqlalchemy import create_engine

engine = create_engine(
    r"sqlite:///C:/Users/lenovo/OneDrive/Desktop/Data_Analyst_Intern/bluestock_mf_capstone/Data/db/bluestock_mf.db"
)

print(engine)

Engine(sqlite:///C:/Users/lenovo/OneDrive/Desktop/Data_Analyst_Intern/bluestock_mf_capstone/Data/db/bluestock_mf.db)


In [5]:
from sqlalchemy import create_engine

engine = create_engine(
    r"sqlite:///C:/Users/lenovo/OneDrive/Desktop/Data_Analyst_Intern/bluestock_mf_capstone/Data/db/bluestock_mf.db"
)

print("Database connected successfully")

Database connected successfully


In [6]:
import pandas as pd

df = pd.read_csv(r"C:\Users\lenovo\OneDrive\Desktop\Data_Analyst_Intern\bluestock_mf_capstone\Data\processed\nav_history_clean.csv")
df2 = pd.read_csv(r"C:\Users\lenovo\OneDrive\Desktop\Data_Analyst_Intern\bluestock_mf_capstone\Data\processed\investor_transactions_clean.csv")
df3 = pd.read_csv(r"C:\Users\lenovo\OneDrive\Desktop\Data_Analyst_Intern\bluestock_mf_capstone\Data\processed\scheme_performance_clean.csv")

In [7]:
dim_fund = df3[
    ['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan']
].drop_duplicates()

dim_fund.head()

,amfi_code,scheme_name,fund_house,category,plan
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular


In [8]:
dim_fund.to_sql(
    'dim_fund',
    engine,
    if_exists='replace',
    index=False
)

print("dim_fund loaded")

dim_fund loaded


In [9]:
print(dim_fund.shape)

(40, 5)


In [10]:
dates1 = pd.DataFrame({
    'full_date': pd.to_datetime(df['date'])
})

dates2 = pd.DataFrame({
    'full_date': pd.to_datetime(df2['transaction_date'])
})

dim_date = pd.concat([dates1, dates2])

dim_date = dim_date.drop_duplicates()

dim_date['year'] = dim_date['full_date'].dt.year
dim_date['quarter'] = dim_date['full_date'].dt.quarter
dim_date['month'] = dim_date['full_date'].dt.month
dim_date['day'] = dim_date['full_date'].dt.day

dim_date = dim_date.reset_index(drop=True)

dim_date['date_id'] = dim_date.index + 1

dim_date.head()

,full_date,year,quarter,month,day,date_id
0,2022-01-03,2022,1,1,3,1
1,2022-01-04,2022,1,1,4,2
2,2022-01-05,2022,1,1,5,3
3,2022-01-06,2022,1,1,6,4
4,2022-01-07,2022,1,1,7,5


In [11]:
dim_date.to_sql(
    'dim_date',
    engine,
    if_exists='replace',
    index=False
)

print("dim_date loaded")

dim_date loaded


In [12]:
print(dim_date.shape)

(1296, 6)


In [13]:
fact_aum = df3[
    ['amfi_code', 'aum_crore']
]

fact_aum.head()

,amfi_code,aum_crore
0,119551,14288
1,119552,1231
2,119598,19259
3,119599,36061
4,119120,24101


In [14]:
fact_aum.to_sql(
    'fact_aum',
    engine,
    if_exists='replace',
    index=False
)

print("fact_aum loaded")

fact_aum loaded


In [15]:
df.to_sql(
    'fact_nav',
    engine,
    if_exists='replace',
    index=False
)

df2.to_sql(
    'fact_transactions',
    engine,
    if_exists='replace',
    index=False
)

df3.to_sql(
    'fact_performance',
    engine,
    if_exists='replace',
    index=False
)

print("All fact tables loaded")

All fact tables loaded


In [16]:
import pandas as pd

pd.read_sql(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    """,
    engine
)

,name
0,dim_fund
1,dim_date
2,fact_aum
3,fact_nav
4,fact_transactions
5,fact_performance


In [17]:
print("dim_fund:", len(dim_fund))
print("dim_date:", len(dim_date))
print("fact_aum:", len(fact_aum))
print("fact_nav:", len(df))
print("fact_transactions:", len(df2))
print("fact_performance:", len(df3))

dim_fund: 40
dim_date: 1296
fact_aum: 40
fact_nav: 46000
fact_transactions: 32778
fact_performance: 40


In [18]:
import pandas as pd

tables = [
    'dim_fund',
    'dim_date',
    'fact_aum',
    'fact_nav',
    'fact_transactions',
    'fact_performance'
]

for table in tables:
    count = pd.read_sql(
        f"SELECT COUNT(*) as rows FROM {table}",
        engine
    )
    print(table)
    print(count)

dim_fund
   rows
0    40
dim_date
   rows
0  1296
fact_aum
   rows
0    40
fact_nav
    rows
0  46000
fact_transactions
    rows
0  32778
fact_performance
   rows
0    40


In [21]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine(
    r"sqlite:///C:/Users/lenovo/OneDrive/Desktop/Data_Analyst_Intern/bluestock_mf_capstone/data/db/bluestock_mf.db"
)

query = """
SELECT
    amfi_code,
    aum_crore
FROM fact_aum
ORDER BY aum_crore DESC
LIMIT 5;
"""

pd.read_sql(query, engine)

,amfi_code,aum_crore
0,148568,49046
1,120842,47469
2,118634,43630
3,149322,41828
4,102886,41728


In [28]:
query = """
SELECT 
    d.fund_house,
    SUM(f.aum_crore) AS total_aum
FROM fact_aum f
JOIN dim_fund d
    ON f.amfi_code = d.amfi_code
GROUP BY d.fund_house
ORDER BY total_aum DESC;
"""

pd.read_sql(query, engine)

,fund_house,total_aum
0,Nippon India MF,154328
1,Kotak Mahindra MF,145689
2,ICICI Prudential MF,120241
3,DSP Mutual Fund,114787
4,Aditya Birla Sun Life MF,104108
5,SBI Mutual Fund,94940
6,Axis Mutual Fund,92210
7,HDFC Mutual Fund,86975
8,UTI Mutual Fund,66990
9,Mirae Asset MF,63396


In [29]:
tables = pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
""", engine)

print(tables)

                name
0           dim_fund
1           dim_date
2           fact_aum
3           fact_nav
4  fact_transactions
5   fact_performance


In [30]:
print(pd.read_sql("PRAGMA table_info(fact_aum)", engine))
print(pd.read_sql("PRAGMA table_info(fact_nav)", engine))
print(pd.read_sql("PRAGMA table_info(fact_transactions)", engine))
print(pd.read_sql("PRAGMA table_info(fact_performance)", engine))

   cid       name    type  notnull dflt_value  pk
0    0  amfi_code  BIGINT        0       None   0
1    1  aum_crore  BIGINT        0       None   0
   cid       name    type  notnull dflt_value  pk
0    0  amfi_code  BIGINT        0       None   0
1    1       date    TEXT        0       None   0
2    2        nav   FLOAT        0       None   0
    cid                name    type  notnull dflt_value  pk
0     0         investor_id    TEXT        0       None   0
1     1    transaction_date    TEXT        0       None   0
2     2           amfi_code  BIGINT        0       None   0
3     3    transaction_type    TEXT        0       None   0
4     4          amount_inr  BIGINT        0       None   0
5     5               state    TEXT        0       None   0
6     6                city    TEXT        0       None   0
7     7           city_tier    TEXT        0       None   0
8     8           age_group    TEXT        0       None   0
9     9              gender    TEXT        0      

In [31]:
import pandas as pd

pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
""", engine)

,name
0,dim_fund
1,dim_date
2,fact_aum
3,fact_nav
4,fact_transactions
5,fact_performance


In [48]:
import pandas as pd
from sqlalchemy import create_engine

# ==========================================
# CONNECT SQLITE DATABASE
# ==========================================

engine = create_engine(
    r"sqlite:///C:/Users/lenovo/OneDrive/Desktop/Data_Analyst_Intern/bluestock_mf_capstone/Data/db/bluestock_mf.db"
)

# ==========================================
# SQL QUERIES
# ==========================================

In [49]:
queries = {

    "Q1_Top5_AUM": """
    SELECT
        amfi_code,
        MAX(aum_crore) AS max_aum
    FROM fact_aum
    GROUP BY amfi_code
    ORDER BY max_aum DESC
    LIMIT 5;
    """,

    "Q2_Average_NAV": """
    SELECT
        amfi_code,
        AVG(nav) AS avg_nav
    FROM fact_nav
    GROUP BY amfi_code
    ORDER BY avg_nav DESC;
    """,

    "Q3_Highest_NAV": """
    SELECT
        amfi_code,
        MAX(nav) AS highest_nav
    FROM fact_nav
    GROUP BY amfi_code
    ORDER BY highest_nav DESC
    LIMIT 10;
    """,

    "Q4_Lowest_NAV": """
    SELECT
        amfi_code,
        MIN(nav) AS lowest_nav
    FROM fact_nav
    GROUP BY amfi_code
    ORDER BY lowest_nav ASC
    LIMIT 10;
    """,

    "Q5_Transaction_Count": """
    SELECT
        transaction_type,
        COUNT(*) AS total_transactions
    FROM fact_transactions
    GROUP BY transaction_type
    ORDER BY total_transactions DESC;
    """,

    "Q6_Amount_By_Type": """
    SELECT
        transaction_type,
        SUM(amount) AS total_amount
    FROM fact_transactions
    GROUP BY transaction_type
    ORDER BY total_amount DESC;
    """,

    "Q7_Expense_Below_1": """
    SELECT
        scheme_name,
        expense_ratio
    FROM fact_performance
    WHERE expense_ratio < 1
    ORDER BY expense_ratio;
    """,

    "Q8_Avg_Expense_Category": """
    SELECT
        category,
        AVG(expense_ratio) AS avg_expense_ratio
    FROM fact_performance
    GROUP BY category
    ORDER BY avg_expense_ratio;
    """,

    "Q9_FundHouse_Count": """
    SELECT
        fund_house,
        COUNT(*) AS total_funds
    FROM dim_fund
    GROUP BY fund_house
    ORDER BY total_funds DESC;
    """,

    "Q10_Category_Count": """
    SELECT
        category,
        COUNT(*) AS total_funds
    FROM dim_fund
    GROUP BY category
    ORDER BY total_funds DESC;
    """
}


In [50]:

# ==========================================
# EXPORT RESULTS TO EXCEL
# ==========================================

output_file = r"C:/Users/lenovo/OneDrive/Desktop/Data_Analyst_Intern/bluestock_mf_capstone/sql_query_outputs.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    for sheet_name, query in queries.items():

        try:
            result = pd.read_sql(query, engine)

            result.to_excel(
                writer,
                sheet_name=sheet_name[:31],  # Excel sheet name limit
                index=False
            )

            print(f"✓ {sheet_name} exported ({len(result)} rows)")

        except Exception as e:
            print(f"✗ Error in {sheet_name}")
            print(e)

print("\nExcel file created successfully!")
print(output_file)

✓ Q1_Top5_AUM exported (5 rows)
✓ Q2_Average_NAV exported (40 rows)
✓ Q3_Highest_NAV exported (10 rows)
✓ Q4_Lowest_NAV exported (10 rows)
✓ Q5_Transaction_Count exported (3 rows)
✗ Error in Q6_Amount_By_Type
(sqlite3.OperationalError) no such column: amount
[SQL: 
    SELECT
        transaction_type,
        SUM(amount) AS total_amount
    FROM fact_transactions
    GROUP BY transaction_type
    ORDER BY total_amount DESC;
    ]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
✗ Error in Q7_Expense_Below_1
(sqlite3.OperationalError) no such column: expense_ratio
[SQL: 
    SELECT
        scheme_name,
        expense_ratio
    FROM fact_performance
    WHERE expense_ratio < 1
    ORDER BY expense_ratio;
    ]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
✗ Error in Q8_Avg_Expense_Category
(sqlite3.OperationalError) no such column: expense_ratio
[SQL: 
    SELECT
        category,
        AVG(expense_ratio) AS avg_expense_ratio
    FROM fact_performance
    GR